# 04 — Train Models

Merges `stock_prices` + `daily_sentiment`, engineers features, and trains:
- 3 baseline models (Logistic Regression, Decision Tree, Random Forest)
- XGBoost binary classifiers (1-day, 5-day direction)
- XGBoost 3-class classifiers (bearish/neutral/bullish, 1-day and 5-day)
- XGBoost regressors (predicted % return, 1-day and 5-day)
- Per-stock XGBoost models on strong-move days only (1-day and 5-day)

All artifacts save to `../ml_models/` — the exact folder your FastAPI backend already loads from (`app/core/ml_loader.py`), so once this notebook finishes, the API's `/predictions/{ticker}` and `/metrics/*` endpoints work with zero extra config.

Run this after `03_score_sentiment_and_combine.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import sys
sys.path.append('..')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error

from utils.db import get_connection

print("Imports done.")

Imports done.


## Load and merge stock_prices + daily_sentiment

In [2]:
conn = get_connection()
df_stock = pd.read_sql("SELECT * FROM stock_prices", conn)
df_sent = pd.read_sql("SELECT * FROM daily_sentiment", conn)
conn.close()

df_stock['date'] = pd.to_datetime(df_stock['date'])
df_sent['date'] = pd.to_datetime(df_sent['date'])

print(f"Stock rows     : {len(df_stock):,}")
print(f"Sentiment rows : {len(df_sent):,}")

C:\Users\acer\AppData\Local\Temp\ipykernel_12448\2352980921.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_stock = pd.read_sql("SELECT * FROM stock_prices", conn)
C:\Users\acer\AppData\Local\Temp\ipykernel_12448\2352980921.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sent = pd.read_sql("SELECT * FROM daily_sentiment", conn)


Stock rows     : 140,511
Sentiment rows : 27,598


In [3]:
trading_days = df_stock[['ticker', 'date']].copy()

def match_to_next_trading_day(sent_df, stock_df):
    results = []
    for _, row in sent_df.iterrows():
        ticker = row['ticker']
        news_date = row['date']
        ticker_days = stock_df[stock_df['ticker'] == ticker]['date']
        window = ticker_days[
            (ticker_days > news_date) &
            (ticker_days <= news_date + pd.Timedelta(days=3))
        ]
        if len(window) > 0:
            results.append({
                'ticker': ticker,
                'matched_date': window.min(),
                'article_count': row['article_count'],
                'avg_compound': row['avg_compound'],
                'max_compound': row.get('max_compound', None),
                'min_compound': row.get('min_compound', None),
                'std_compound': row.get('std_compound', None),
                'positive_count': row['positive_count'],
                'negative_count': row['negative_count'],
            })
    return pd.DataFrame(results)

print("Matching...")
matched = match_to_next_trading_day(df_sent, trading_days)

merged = pd.merge(
    df_stock,
    matched.rename(columns={'matched_date': 'date'})[
        ['ticker', 'date', 'avg_compound', 'max_compound',
         'min_compound', 'std_compound', 'article_count',
         'positive_count', 'negative_count']
    ],
    on=['ticker', 'date'],
    how='inner'
)
merged = merged.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f"Merged rows : {len(merged):,}")

Matching...
Merged rows : 26,955


## Feature engineering

In [4]:
# Sentiment centering — removes positive bias, uses relative sentiment per stock
merged['avg_compound_centered'] = (
    merged.groupby('ticker')['avg_compound'].transform(lambda x: x - x.mean())
)
merged['sentiment_3d_centered'] = (
    merged.groupby('ticker')['avg_compound_centered']
          .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
merged['sentiment_momentum'] = merged['avg_compound_centered'] - merged['sentiment_3d_centered']

# DMA crossovers
merged['price_above_dma20'] = (merged['close_price'] > merged['dma_20']).astype(int)
merged['price_above_dma50'] = (merged['close_price'] > merged['dma_50']).astype(int)
merged['dma20_above_dma50'] = (merged['dma_20'] > merged['dma_50']).astype(int)

# RSI zones
merged['rsi_oversold'] = (merged['rsi'] < 30).astype(int)
merged['rsi_overbought'] = (merged['rsi'] > 70).astype(int)

merged['volume_spike'] = merged['volume_spike'].astype(int)

# MACD crossover signal
merged['macd_bullish'] = (
    (merged['macd'] > merged['macd_signal']) &
    (merged.groupby('ticker')['macd'].shift(1) <= merged.groupby('ticker')['macd_signal'].shift(1))
).astype(int)
merged['macd_bearish'] = (
    (merged['macd'] < merged['macd_signal']) &
    (merged.groupby('ticker')['macd'].shift(1) >= merged.groupby('ticker')['macd_signal'].shift(1))
).astype(int)
merged['macd_hist_positive'] = (merged['macd_hist'] > 0).astype(int)

# Bollinger band position
merged['bb_position'] = merged['bb_position'].clip(0, 1)
merged['bb_oversold'] = (merged['bb_position'] < 0.2).astype(int)
merged['bb_overbought'] = (merged['bb_position'] > 0.8).astype(int)

print("Features engineered.")

Features engineered.


## Build targets (1-day and 5-day direction, binary + 3-class)

In [5]:
merged = merged.sort_values(['ticker', 'date']).reset_index(drop=True)

merged['future_close_1d'] = merged.groupby('ticker')['close_price'].shift(-1)
merged['future_return_1d'] = (
    (merged['future_close_1d'] - merged['close_price']) / merged['close_price'] * 100
)
merged['future_close_5d'] = merged.groupby('ticker')['close_price'].shift(-5)
merged['future_return_5d'] = (
    (merged['future_close_5d'] - merged['close_price']) / merged['close_price'] * 100
)

# 3-class target: 2=BULLISH (>+1%), 0=BEARISH (<-1%), 1=NEUTRAL
def make_3class(returns):
    return np.where(returns > 1.0, 2, np.where(returns < -1.0, 0, 1))

merged['target_1d_3class'] = make_3class(merged['future_return_1d'])
merged['target_5d_3class'] = make_3class(merged['future_return_5d'])
merged['target_1d'] = (merged['future_close_1d'] > merged['close_price']).astype(int)
merged['target_5d'] = (merged['future_close_5d'] > merged['close_price']).astype(int)

merged = merged.dropna(subset=[
    'future_close_1d', 'future_close_5d', 'dma_20', 'dma_50', 'rsi',
    'macd', 'macd_signal', 'macd_hist', 'bb_position', 'return_1d'
])
merged = merged.reset_index(drop=True)

print(f"Final rows : {len(merged):,}")
print(f"\n1D distribution:")
print(merged['target_1d_3class'].value_counts().sort_index().rename({0: 'BEARISH', 1: 'NEUTRAL', 2: 'BULLISH'}))

Final rows : 26,695

1D distribution:
target_1d_3class
BEARISH     6774
NEUTRAL    11858
BULLISH     8063
Name: count, dtype: int64


## Train/test split — chronological (by date, not random)

In [6]:
FEATURES = [
    'avg_compound_centered', 'sentiment_3d_centered', 'sentiment_momentum',
    'article_count', 'positive_count', 'negative_count',
    'max_compound', 'min_compound', 'std_compound',
    'rsi', 'rsi_oversold', 'rsi_overbought',
    'price_above_dma20', 'price_above_dma50', 'dma20_above_dma50',
    'macd', 'macd_signal', 'macd_hist', 'macd_bullish', 'macd_bearish', 'macd_hist_positive',
    'bb_position', 'bb_oversold', 'bb_overbought',
    'return_1d', 'return_3d', 'return_5d',
    'pct_from_20d_high', 'pct_from_20d_low',
    'volume_spike',
]

merged = merged.sort_values(['date', 'ticker']).reset_index(drop=True)
X = merged[FEATURES]

unique_dates = sorted(merged['date'].unique())
split_date = unique_dates[int(len(unique_dates) * 0.8)]
split = (merged['date'] <= split_date).sum()

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train_1d, y_test_1d = merged['target_1d'].iloc[:split], merged['target_1d'].iloc[split:]
y_train_1d_ret, y_test_1d_ret = merged['future_return_1d'].iloc[:split], merged['future_return_1d'].iloc[split:]
y_train_5d, y_test_5d = merged['target_5d'].iloc[:split], merged['target_5d'].iloc[split:]
y_train_5d_ret, y_test_5d_ret = merged['future_return_5d'].iloc[:split], merged['future_return_5d'].iloc[split:]
y_train_1d_3c, y_test_1d_3c = merged['target_1d_3class'].iloc[:split], merged['target_1d_3class'].iloc[split:]
y_train_5d_3c, y_test_5d_3c = merged['target_5d_3class'].iloc[:split], merged['target_5d_3class'].iloc[split:]

print(f"Train rows: {len(X_train):,}  Test rows: {len(X_test):,}")
print(f"Split date: {split_date.date()}")

Train rows: 19,406  Test rows: 7,289
Split date: 2023-06-20


## Model 1 — Logistic Regression (baseline)

In [27]:
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train_1d)
pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test_1d, pred_lr)
print(f"1 Day Accuracy : {acc_lr*100:.2f}%")
print(classification_report(y_test_1d, pred_lr, target_names=['DOWN', 'UP']))

1 Day Accuracy : 52.78%
              precision    recall  f1-score   support

        DOWN       0.54      0.63      0.58      3761
          UP       0.52      0.42      0.46      3528

    accuracy                           0.53      7289
   macro avg       0.53      0.52      0.52      7289
weighted avg       0.53      0.53      0.52      7289



C:\Users\acer\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Model 2 — Decision Tree

In [28]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train_1d)
pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test_1d, pred_dt)
print(f"1 Day Accuracy : {acc_dt*100:.2f}%")
print(classification_report(y_test_1d, pred_dt, target_names=['DOWN', 'UP']))

1 Day Accuracy : 51.64%
              precision    recall  f1-score   support

        DOWN       0.53      0.56      0.54      3761
          UP       0.50      0.47      0.49      3528

    accuracy                           0.52      7289
   macro avg       0.52      0.51      0.51      7289
weighted avg       0.52      0.52      0.52      7289



## Model 3 — Random Forest

In [29]:
rf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train_1d)
pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test_1d, pred_rf)
print(f"1 Day Accuracy : {acc_rf*100:.2f}%")
print(classification_report(y_test_1d, pred_rf, target_names=['DOWN', 'UP']))

1 Day Accuracy : 52.35%
              precision    recall  f1-score   support

        DOWN       0.52      0.83      0.64      3761
          UP       0.52      0.20      0.29      3528

    accuracy                           0.52      7289
   macro avg       0.52      0.51      0.46      7289
weighted avg       0.52      0.52      0.47      7289



## Model 4 — XGBoost binary (1-day and 5-day)

In [30]:
xgb_params = dict(
    max_depth=3, subsample=0.6, colsample_bytree=0.6,
    min_child_weight=10, gamma=1.5, reg_alpha=2, reg_lambda=5,
    random_state=42, eval_metric='logloss', verbosity=0
)

clf_1d = XGBClassifier(n_estimators=200, learning_rate=0.03, **xgb_params)
clf_1d.fit(X_train, y_train_1d, eval_set=[(X_test, y_test_1d)], verbose=False)
pred_1d = clf_1d.predict(X_test)
acc_1d = accuracy_score(y_test_1d, pred_1d)
print(f"1 Day Accuracy : {acc_1d*100:.2f}%")

clf_5d = XGBClassifier(n_estimators=100, learning_rate=0.04, **xgb_params)
clf_5d.fit(X_train, y_train_5d, eval_set=[(X_test, y_test_5d)], verbose=False)
pred_5d = clf_5d.predict(X_test)
acc_5d = accuracy_score(y_test_5d, pred_5d)
print(f"5 Day Accuracy : {acc_5d*100:.2f}%")

1 Day Accuracy : 53.15%
5 Day Accuracy : 58.20%


## Model 5 — XGBoost 3-class (bearish/neutral/bullish)

In [31]:
clf_1d_3c = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, random_state=42, num_class=3,
    eval_metric='mlogloss', verbosity=0
)
clf_1d_3c.fit(X_train, y_train_1d_3c, eval_set=[(X_test, y_test_1d_3c)], verbose=False)
acc_1d_3c = accuracy_score(y_test_1d_3c, clf_1d_3c.predict(X_test))
print(f"1 Day 3-Class Accuracy : {acc_1d_3c*100:.2f}%")

clf_5d_3c = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, random_state=42, num_class=3,
    eval_metric='mlogloss', verbosity=0
)
clf_5d_3c.fit(X_train, y_train_5d_3c, eval_set=[(X_test, y_test_5d_3c)], verbose=False)
acc_5d_3c = accuracy_score(y_test_5d_3c, clf_5d_3c.predict(X_test))
print(f"5 Day 3-Class Accuracy : {acc_5d_3c*100:.2f}%")

1 Day 3-Class Accuracy : 49.22%
5 Day 3-Class Accuracy : 46.43%


## XGBoost regressors — predicted % return

In [32]:
reg_1d = XGBRegressor(n_estimators=200, learning_rate=0.03, **xgb_params)
reg_1d.fit(X_train, y_train_1d_ret, eval_set=[(X_test, y_test_1d_ret)], verbose=False)

reg_5d = XGBRegressor(n_estimators=100, learning_rate=0.04, **xgb_params)
reg_5d.fit(X_train, y_train_5d_ret, eval_set=[(X_test, y_test_5d_ret)], verbose=False)

mae_1d = mean_absolute_error(y_test_1d_ret, reg_1d.predict(X_test))
mae_5d = mean_absolute_error(y_test_5d_ret, reg_5d.predict(X_test))
print(f"1 Day MAE : {mae_1d:.2f}%")
print(f"5 Day MAE : {mae_5d:.2f}%")

1 Day MAE : 1.75%
5 Day MAE : 4.57%


## Model comparison summary

In [33]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest',
              'XGBoost 1D Binary', 'XGBoost 5D Binary',
              'XGBoost 1D 3-Class', 'XGBoost 5D 3-Class'],
    'Accuracy': [f"{a*100:.2f}%" for a in
                 [acc_lr, acc_dt, acc_rf, acc_1d, acc_5d, acc_1d_3c, acc_5d_3c]],
})
print(comparison_df.to_string(index=False))

model_comparison = {
    'Logistic Regression': acc_lr,
    'Decision Tree': acc_dt,
    'Random Forest': acc_rf,
    'XGBoost 1 Day': acc_1d,
    'XGBoost 5 Day': acc_5d,
}

              Model Accuracy
Logistic Regression   52.78%
      Decision Tree   51.64%
      Random Forest   52.35%
  XGBoost 1D Binary   53.15%
  XGBoost 5D Binary   58.20%
 XGBoost 1D 3-Class   49.22%
 XGBoost 5D 3-Class   46.43%


## Save global models to ml_models/

In [34]:
os.makedirs('../ml_models', exist_ok=True)

joblib.dump(lr, '../ml_models/logistic_regression.pkl')
joblib.dump(dt, '../ml_models/decision_tree.pkl')
joblib.dump(rf, '../ml_models/random_forest.pkl')
joblib.dump(clf_1d, '../ml_models/xgb_clf_1d.pkl')
joblib.dump(clf_5d, '../ml_models/xgb_clf_5d.pkl')
joblib.dump(reg_1d, '../ml_models/xgb_reg_1d.pkl')
joblib.dump(reg_5d, '../ml_models/xgb_reg_5d.pkl')
joblib.dump(FEATURES, '../ml_models/features.pkl')
joblib.dump(model_comparison, '../ml_models/model_comparison.pkl')

print("Global models saved.")
print(sorted(os.listdir('../ml_models')))

Global models saved.
['.gitkeep', 'decision_tree.pkl', 'features.pkl', 'logistic_regression.pkl', 'model_comparison.pkl', 'per_stock_1d', 'per_stock_5d', 'random_forest.pkl', 'xgb_clf_1d.pkl', 'xgb_clf_5d.pkl', 'xgb_reg_1d.pkl', 'xgb_reg_5d.pkl']


## Per-stock models — 5-day, strong moves only (>1.5% or <-1.5%)

In [35]:
os.makedirs('../ml_models/per_stock_5d', exist_ok=True)
per_stock_5d_results = {}

for ticker in sorted(merged['ticker'].unique()):
    stock = merged[merged['ticker'] == ticker].copy().reset_index(drop=True)
    stock_5d = stock[
        (stock['future_return_5d'] > 1.5) | (stock['future_return_5d'] < -1.5)
    ].copy().reset_index(drop=True)
    stock_5d['target_strong_5d'] = (stock_5d['future_return_5d'] > 0).astype(int)

    X_s, y_s = stock_5d[FEATURES], stock_5d['target_strong_5d']
    split_s = int(len(stock_5d) * 0.70)
    if split_s < 10 or len(stock_5d) - split_s < 5:
        print(f"{ticker:<15} skipped — not enough strong-move rows ({len(stock_5d)})")
        continue

    X_tr, X_te = X_s.iloc[:split_s], X_s.iloc[split_s:]
    y_tr, y_te = y_s.iloc[:split_s], y_s.iloc[split_s:]

    clf = XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='logloss', verbosity=0
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    acc = accuracy_score(y_te, clf.predict(X_te))
    per_stock_5d_results[ticker] = {'model': clf, 'accuracy': acc, 'rows': len(stock_5d)}

    safe_ticker = ticker.replace('.', '_').replace('&', 'and').replace('-', '_')
    joblib.dump(clf, f'../ml_models/per_stock_5d/{safe_ticker}.pkl')
    print(f"{ticker:<15} 5d acc: {acc*100:.1f}%  rows: {len(stock_5d)}")

ADANIENT.NS     5d acc: 40.1%  rows: 623
ADANIPORTS.NS   5d acc: 40.8%  rows: 236
APOLLOHOSP.NS   5d acc: 63.0%  rows: 179
ASIANPAINT.NS   5d acc: 45.5%  rows: 220
AXISBANK.NS     5d acc: 57.5%  rows: 617
BAJAJ-AUTO.NS   5d acc: 32.3%  rows: 310
BAJAJFINSV.NS   5d acc: 60.5%  rows: 124
BAJFINANCE.NS   5d acc: 47.5%  rows: 264
BHARTIARTL.NS   5d acc: 46.9%  rows: 709
BPCL.NS         5d acc: 40.0%  rows: 298
BRITANNIA.NS    5d acc: 57.6%  rows: 196
CIPLA.NS        5d acc: 49.3%  rows: 227
COALINDIA.NS    5d acc: 62.5%  rows: 291
DIVISLAB.NS     5d acc: 49.0%  rows: 509
DRREDDY.NS      5d acc: 51.4%  rows: 233
EICHERMOT.NS    5d acc: 55.4%  rows: 306
GRASIM.NS       5d acc: 74.2%  rows: 101
HCLTECH.NS      5d acc: 49.0%  rows: 333
HDFCBANK.NS     5d acc: 60.8%  rows: 849
HDFCLIFE.NS     5d acc: 57.1%  rows: 140
HEROMOTOCO.NS   5d acc: 44.7%  rows: 377
HINDALCO.NS     5d acc: 57.3%  rows: 247
HINDUNILVR.NS   5d acc: 45.8%  rows: 480
ICICIBANK.NS    5d acc: 49.6%  rows: 806
INDUSINDBK.NS   

## Per-stock models — 1-day, strong moves only (>1% or <-1%)

In [36]:
os.makedirs('../ml_models/per_stock_1d', exist_ok=True)
per_stock_1d_results = {}

for ticker in sorted(merged['ticker'].unique()):
    stock = merged[merged['ticker'] == ticker].copy().reset_index(drop=True)
    stock_1d = stock[
        (stock['future_return_1d'] > 1.0) | (stock['future_return_1d'] < -1.0)
    ].copy().reset_index(drop=True)
    stock_1d['target_strong_1d'] = (stock_1d['future_return_1d'] > 0).astype(int)

    X_s, y_s = stock_1d[FEATURES], stock_1d['target_strong_1d']
    split_s = int(len(stock_1d) * 0.70)
    if split_s < 10 or len(stock_1d) - split_s < 5:
        print(f"{ticker:<15} skipped — not enough strong-move rows ({len(stock_1d)})")
        continue

    X_tr, X_te = X_s.iloc[:split_s], X_s.iloc[split_s:]
    y_tr, y_te = y_s.iloc[:split_s], y_s.iloc[split_s:]

    clf = XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='logloss', verbosity=0
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    acc = accuracy_score(y_te, clf.predict(X_te))
    per_stock_1d_results[ticker] = {'model': clf, 'accuracy': acc, 'rows': len(stock_1d)}

    safe_ticker = ticker.replace('.', '_').replace('&', 'and').replace('-', '_')
    joblib.dump(clf, f'../ml_models/per_stock_1d/{safe_ticker}.pkl')
    print(f"{ticker:<15} 1d acc: {acc*100:.1f}%  rows: {len(stock_1d)}")

ADANIENT.NS     1d acc: 52.6%  rows: 447
ADANIPORTS.NS   1d acc: 42.9%  rows: 184
APOLLOHOSP.NS   1d acc: 53.7%  rows: 134
ASIANPAINT.NS   1d acc: 51.0%  rows: 167
AXISBANK.NS     1d acc: 50.0%  rows: 491
BAJAJ-AUTO.NS   1d acc: 51.4%  rows: 237
BAJAJFINSV.NS   1d acc: 50.0%  rows: 99
BAJFINANCE.NS   1d acc: 57.1%  rows: 210
BHARTIARTL.NS   1d acc: 45.7%  rows: 539
BPCL.NS         1d acc: 42.5%  rows: 243
BRITANNIA.NS    1d acc: 53.3%  rows: 147
CIPLA.NS        1d acc: 42.3%  rows: 172
COALINDIA.NS    1d acc: 59.0%  rows: 257
DIVISLAB.NS     1d acc: 50.5%  rows: 369
DRREDDY.NS      1d acc: 52.7%  rows: 183
EICHERMOT.NS    1d acc: 46.1%  rows: 252
GRASIM.NS       1d acc: 37.5%  rows: 78
HCLTECH.NS      1d acc: 42.3%  rows: 234
HDFCBANK.NS     1d acc: 53.5%  rows: 516
HDFCLIFE.NS     1d acc: 42.4%  rows: 110
HEROMOTOCO.NS   1d acc: 49.4%  rows: 275
HINDALCO.NS     1d acc: 57.1%  rows: 207
HINDUNILVR.NS   1d acc: 52.0%  rows: 338
ICICIBANK.NS    1d acc: 46.4%  rows: 559
INDUSINDBK.NS   1d

## Save per-stock result dictionaries (used by /metrics/per-stock-accuracy)

In [37]:
joblib.dump(per_stock_5d_results, '../ml_models/per_stock_5d_results.pkl')
joblib.dump(per_stock_1d_results, '../ml_models/per_stock_1d_results.pkl')
print("Saved per_stock_1d_results.pkl and per_stock_5d_results.pkl")

print("\nAll model training complete. Restart the FastAPI backend (uvicorn app.main:app --reload) to pick up the new models.")

Saved per_stock_1d_results.pkl and per_stock_5d_results.pkl

All model training complete. Restart the FastAPI backend (uvicorn app.main:app --reload) to pick up the new models.
